# PLN Projeto 1

In [148]:
import pandas as pd
import re
import uuid

df_cases = pd.read_csv("../../data/external/cases.csv")
df_meta = pd.read_csv("../../data/external/metadata.csv")
# Cruzamento dos dados: Adiciona metadados aos casos
df_merged = df_cases.merge(df_meta, on='article_id', how='left')

In [ ]:
token_re = re.compile(r"""
    \d+(?:\.\d+)?(?:\s*x\s*10\d*)?
    |[a-zA-ZÀ-ÿ]+
    |[.,;:!?()%/]
""", re.VERBOSE)

def tokenize(sent):
    return [(m.group(), m.start(), m.end()) for m in token_re.finditer(sent)]

In [150]:
def normalize(tok):
    return tok.lower().rstrip('sáéíóú')  # naive, mas cobre plural/acento simples

In [151]:
def build_gazetteers():
    return {
        'Diagnosis': ['diabetes mellitus', 'hypertension', 'acute pancreatitis', 'chronic pancreatitis', 'pancreatic pseudocyst', 'pseudocyst', 'solid mass', 'type 2 diabetes', 'tumor', 'cancer'],
        'Symptom': ['epigastric pain', 'nausea', 'fever', 'epigastric tenderness', 'fat stranding', 'pain', 'headache', 'cough'],
        'Exam': [
          'computed tomography', 'ct', 'thoracic ct', 'endoscopic ultrasound', 'eus', 'ultrasound', 'laboratory tests', 'mri', 'x-ray', 'echocardiogram', 'echocardiography', 'ecg', 'flow cytometry',
          'hemoglobin', 'hematocrit', 'white blood cell count', 'total leukocyte count', 'leucocytosis', 'lymphocyte count', 'platelet count', 'platelet counts', 'mean corpuscular volume', 
          'reticulocyte count', 'haptoglobin', 'adamts13 activity', 'c-reactive protein', 'crp', 'erythrocyte sedimentation rate', 'esr', 'creatinine', 'serum creatinine', 'albumin', 'total protein',
          'ferritin', 'lactate dehydrogenase', 'ldh', 'complement c3', 'complement c4', 'c3', 'c4', 'troponin', 'troponin i', 'ck-mb', 'creatine kinase', 'myoglobin', 'ejection fraction', 'lvef', 
          'mean gradient', 'nt-probnp', 'heart rate', 'blood pressure', 'mean arterial blood pressure', 'glycosylated hemoglobin', 'hemoglobin a1c', 'hba1c', 'blood sugar', 'immunoglobulin m', 
          'immunoglobulin g', 'igm', 'igg', 'igg levels', 'forced vital capacity', 'fvc', 'forced expiratory volume', 'fev1', 'diffusing capacity', 'dlco', 'oxygen saturation', 'respiratory rate', 
          'carcinoembryonic antigen', 'cea', 'pd-l1 tps score','hydroxychloroquine level', 'gtt',
      ],
        'Treatment': ['intravenous fluids', 'analgesia', 'cystgastrostomy', 'drainage', 'surgery', 'chemotherapy']
    }

In [177]:
def build_lookup(gazetteer):
    lookup = {}
    for ent_type, keywords in gazetteer.items():
        for kw in keywords:
            for tok in tokenize(kw):
                lookup[normalize(tok[0])] = (ent_type, kw)
    print(lookup)
    return lookup

In [178]:
gazetteer = build_gazetteers()
lookup = build_lookup(gazetteer)

{'diabete': ('Diagnosis', 'type 2 diabetes'), 'mellitu': ('Diagnosis', 'diabetes mellitus'), 'hypertension': ('Diagnosis', 'hypertension'), 'acute': ('Diagnosis', 'acute pancreatitis'), 'pancreatiti': ('Diagnosis', 'chronic pancreatitis'), 'chronic': ('Diagnosis', 'chronic pancreatitis'), 'pancreatic': ('Diagnosis', 'pancreatic pseudocyst'), 'pseudocyst': ('Diagnosis', 'pseudocyst'), 'solid': ('Diagnosis', 'solid mass'), 'ma': ('Diagnosis', 'solid mass'), 'type': ('Diagnosis', 'type 2 diabetes'), '2': ('Diagnosis', 'type 2 diabetes'), 'tumor': ('Diagnosis', 'tumor'), 'cancer': ('Diagnosis', 'cancer'), 'epigastric': ('Symptom', 'epigastric tenderness'), 'pain': ('Symptom', 'pain'), 'nausea': ('Symptom', 'nausea'), 'fever': ('Symptom', 'fever'), 'tenderne': ('Symptom', 'epigastric tenderness'), 'fat': ('Symptom', 'fat stranding'), 'stranding': ('Symptom', 'fat stranding'), 'headache': ('Symptom', 'headache'), 'cough': ('Symptom', 'cough'), 'computed': ('Exam', 'computed tomography'), 'to

In [154]:
def create_node(all_nodes, node_id, node_type, node_label, **attributes):
    node = {
        "node_id": node_id,
        "type": node_type,
        "label": node_label,
        "attributes": attributes,
    }
    all_nodes.append(node)
    return node

In [155]:
def create_edge(all_edges, edge_id, source_id, target_id, edge_type, relation, **attributes):
    edge = {
        "edge_id": edge_id,
        "source": source_id,
        "target": target_id,
        "type": edge_type,
        "relation": relation,
        "attributes": attributes,
    }
    all_edges.append(edge)
    return edge

In [ ]:
def process_multicare_dataset(row, lookup):
    all_nodes = []
    all_edges = []
    seen_edges = set()

    case_id = row["case_id"]
    text = row["case_text"]

    # Creates patient node
    patient_id = f"P_{case_id}"
    age = row.get('age', 'Unknown')
    gender = row.get('gender', 'Unknown')
    create_node(all_nodes, patient_id, 'Patient', f'Case {case_id}', age=age, gender=gender)

    entity_map = {}
    valid_units = {'u/l', 'mg/l', 'ng/ml', 'mmol/l', 'g/dl', '%', 'au/ml', 'mmhg', 'l'}
    # Regex for numbers and scientific notation
    num_re = re.compile(r'\d+(?:\.\d+)?(?:\s*x\s*10\d*)?')

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sent in sentences:
        tokens = tokenize(sent)
        last_node_id = patient_id
        last_type = None

        for i, (tok, start, end) in enumerate(tokens):
            key = normalize(tok)
            if key not in lookup:
                continue
            ent_type, keyword = lookup[key]

            if keyword not in entity_map:
                node_id = f"{ent_type[:3].upper()}_{uuid.uuid4().hex[:6]}"
                create_node(all_nodes, node_id, ent_type, keyword.capitalize())
                entity_map[keyword] = node_id

            node_id = entity_map[keyword]

            relation = 'ASSOCIATED_WITH'
            if ent_type == 'Diagnosis': relation = 'DIAGNOSED_WITH'
            elif ent_type == 'Symptom': relation = 'HAS_SYMPTOM'
            elif ent_type == 'Exam': relation = 'UNDERWENT_EXAM'
            elif ent_type == 'Treatment': relation = 'TREATED_BY'

            edge_key = (patient_id, node_id, relation)
            if edge_key not in seen_edges:
                seen_edges.add(edge_key)
                create_edge(all_edges, f"e_{uuid.uuid4().hex[:6]}",
                            patient_id, node_id, ent_type, relation, source='regex_dict')

            if ent_type == 'Exam':
                last_node_id, last_type = node_id, 'Exam'

                for j in range(i + 1, len(tokens)):
                    next_tok = tokens[j][0]
                    if normalize(next_tok) in lookup:
                        break
                    if num_re.fullmatch(next_tok) and j + 1 < len(tokens):
                        unit_tok = tokens[j + 1][0].lower().lstrip('/')
                        if unit_tok in valid_units:
                            res_id = f"VAL_{uuid.uuid4().hex[:6]}"
                            create_node(all_nodes, res_id, 'ExamResult',
                                        f'{next_tok} {unit_tok}', value=next_tok, unit=unit_tok)

                            exam_edge_key = (last_node_id, res_id, 'HAS_RESULT')
                            if exam_edge_key not in seen_edges:
                                seen_edges.add(exam_edge_key)
                                create_edge(all_edges, f"e_{uuid.uuid4().hex[:6]}",
                                            last_node_id, res_id, 'ExamResult',
                                            'HAS_RESULT', source='regex_extraction')
                            break

    return pd.DataFrame(all_nodes), pd.DataFrame(all_edges)

In [ ]:
df_nodes, df_edges = process_multicare_dataset(df_merged.iloc[6], lookup)  # Processa apenas o caso de índice 10 como exemplo)
df_nodes.to_csv("../../data/processed/nodes.csv", index=False)
df_edges.to_csv("../../data/processed/edges.csv", index=False)
print(f"Extração em lote concluída! {len(df_nodes)} nós e {len(df_edges)} arestas criadas.")

OSError: Cannot save file into a non-existent directory: '../../processed/data'

In [195]:
def to_mermaid(nodes, edges) -> str:
    lines = ["flowchart LR"]
    for n in nodes.itertuples():
        label = str(n.label).replace('"', "'")
        lines.append(f'  {n.node_id}["{n.type}<br/>{label}"]')
    for e in edges.itertuples():
        lines.append(f'  {e.source} -->|{e.relation}| {e.target}')
    return "\n".join(lines)

In [196]:
from IPython.display import display, Markdown

mermaid_md = to_mermaid(df_nodes, df_edges)
display(Markdown(f"```mermaid\n{mermaid_md}\n```"))

```mermaid
flowchart LR
  P_PMC4835621_01["Patient<br/>Case PMC4835621_01"]
  EXA_4b90c1["Exam<br/>Hemoglobin a1c"]
  SYM_5e84c3["Symptom<br/>Headache"]
  SYM_add7c3["Symptom<br/>Fever"]
  SYM_eb5b18["Symptom<br/>Nausea"]
  DIA_d9dd1b["Diagnosis<br/>Hypertension"]
  EXA_f93aca["Exam<br/>Laboratory tests"]
  EXA_360e5e["Exam<br/>Forced vital capacity"]
  EXA_a12e32["Exam<br/>Heart rate"]
  EXA_e9a012["Exam<br/>Thoracic ct"]
  EXA_d2eaf3["Exam<br/>White blood cell count"]
  EXA_f627b1["Exam<br/>Blood sugar"]
  EXA_d55812["Exam<br/>Reticulocyte count"]
  VAL_0ad14c["ExamResult<br/>73 %"]
  EXA_fe59f4["Exam<br/>Immunoglobulin g"]
  EXA_a65abc["Exam<br/>Mean arterial blood pressure"]
  EXA_b03f84["Exam<br/>Mean corpuscular volume"]
  EXA_9800d3["Exam<br/>Forced expiratory volume"]
  VAL_47cf20["ExamResult<br/>2.4 %"]
  EXA_27c405["Exam<br/>Hydroxychloroquine level"]
  EXA_5b4dfb["Exam<br/>Serum creatinine"]
  EXA_f0602a["Exam<br/>Lactate dehydrogenase"]
  EXA_403d70["Exam<br/>Ldh"]
  EXA_9534a0["Exam<br/>Pd-l1 tps score"]
  EXA_e96ffc["Exam<br/>Haptoglobin"]
  EXA_e0fbce["Exam<br/>Adamts13 activity"]
  VAL_10ea1e["ExamResult<br/>10 %"]
  EXA_fd6956["Exam<br/>Computed tomography"]
  EXA_c4bc1a["Exam<br/>Platelet counts"]
  DIA_aefa97["Diagnosis<br/>Chronic pancreatitis"]
  DIA_8b26ea["Diagnosis<br/>Type 2 diabetes"]
  VAL_515f0c["ExamResult<br/>2.5 %"]
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_4b90c1
  P_PMC4835621_01 -->|PRESENTS_WITH| SYM_5e84c3
  P_PMC4835621_01 -->|PRESENTS_WITH| SYM_add7c3
  P_PMC4835621_01 -->|PRESENTS_WITH| SYM_eb5b18
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_d9dd1b
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_f93aca
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_360e5e
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_a12e32
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_e9a012
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_d2eaf3
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_f627b1
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_d55812
  EXA_d55812 -->|HAS_RESULT| VAL_0ad14c
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_fe59f4
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_a65abc
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_b03f84
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_9800d3
  EXA_d55812 -->|HAS_RESULT| VAL_47cf20
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_27c405
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_5b4dfb
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_f0602a
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_403d70
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_9534a0
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_e96ffc
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_e0fbce
  EXA_f93aca -->|HAS_RESULT| VAL_10ea1e
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_fd6956
  P_PMC4835621_01 -->|UNDERWENT_EXAM| EXA_c4bc1a
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_aefa97
  P_PMC4835621_01 -->|DIAGNOSED_WITH| DIA_8b26ea
  EXA_27c405 -->|HAS_RESULT| VAL_515f0c
```